In [1]:
!pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\ianmu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import wandb
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("WANDB_API_KEY")

if api_key:
    os.environ["WANDB_API_KEY"] = api_key
    wandb.login(key=api_key)
else:
    print("WANDB_API_KEY no está")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ianmu\_netrc
wandb: Currently logged in as: mauu (mauu-tec) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
import torch

# Configurar precisión para Tensor Cores (API nueva)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"TF32 para matmul: {torch.backends.cuda.matmul.allow_tf32}")
print(f"TF32 para cuDNN: {torch.backends.cudnn.allow_tf32}")

GPU: NVIDIA GeForce RTX 3070 Ti
CUDA disponible: True
TF32 para matmul: True
TF32 para cuDNN: True


C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\backends\__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  self.setter(val)


In [7]:
import subprocess
import sys

def run_training(cmd):
    result = subprocess.run(cmd, encoding='utf-8', errors='replace')
    return result.returncode

run_training([sys.executable, "train_model_a.py", "train.lr=1e-3", "dataset.batch_size=32", "train.epochs=50"])
run_training([sys.executable, "train_model_a.py", "train.lr=5e-4", "dataset.batch_size=64", "train.epochs=40"])
run_training([sys.executable, "train_model_a.py", "train.lr=1e-4", "dataset.batch_size=32", "train.epochs=60"])

run_training([sys.executable, "train_model_b.py", "model=distilled", "train.lr=1e-3", "dataset.batch_size=32", "train.epochs=50", "model.distillation.temperature=4.0", "model.distillation.alpha=0.7"])
run_training([sys.executable, "train_model_b.py", "model=distilled", "train.lr=5e-4", "dataset.batch_size=64", "train.epochs=40", "model.distillation.temperature=3.0", "model.distillation.alpha=0.5"])
run_training([sys.executable, "train_model_b.py", "model=distilled", "train.lr=1e-4", "dataset.batch_size=32", "train.epochs=60", "model.distillation.temperature=5.0", "model.distillation.alpha=0.8"])

run_training([sys.executable, "train_model_c.py", "model=unet_ae", "train.lr=1e-3", "dataset.batch_size=32", "train.epochs=50", "model.latent.dim=256", "model.reconstruction.loss=l1"])
run_training([sys.executable, "train_model_c.py", "model=unet_ae", "train.lr=5e-4", "dataset.batch_size=64", "train.epochs=40", "model.latent.dim=128", "model.reconstruction.loss=l2"])
run_training([sys.executable, "train_model_c.py", "model=unet_ae", "train.lr=1e-4", "dataset.batch_size=32", "train.epochs=60", "model.latent.dim=512", "model.reconstruction.loss=l1"])

0

In [4]:
from src.train_model_a import train_model_a

model, trainer = train_model_a(
    lr=1e-3,
    batch_size=16,
    epochs=3,
    embedding_dim=64,
    hidden_dim=32,
    run_name="modeloA_test_run"
)



Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | model     | ResNet18Partial  | 7.1 M  | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
7.1 M     Trainable params
0         Non-trainable params
7.1 M     Total params
28.432    Total estimated model params size (MB)
46        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▅▅██
train/loss,█▂▁
trainer/global_step,▁▁▅▅██
val/loss,█▂▁
epoch,2
train/loss,0.12038
trainer/global_step,380
val/loss,0.00188


In [3]:
from src.train_model_b import train_model_b

model, trainer = train_model_b(
    lr=1e-3,
    batch_size=16,
    epochs=3,
    embedding_dim=64,
    hidden_dim=32,
    run_name="modeloB_test_run",
    temperature=4.0,
    alpha=0.7
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name    | Type             | Params | Mode 
-----------------------------------------------------
0 | student | ResNet18Partial  | 7.1 M  | train
1 | teacher | ResNet           | 11.2 M | eval 
2 | ce_loss | CrossEntropyLoss | 0      | train
3 | kl_loss | KLDivLoss        | 0      | train
-----------------------------------------------------
7.1 M     Trainable params
11.2 M    Non-trainable params
18.3 M    Total params
73.159    Total estimated model params size (MB)
47        Modules in train mode
68        Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
C:\Users\ianmu\AppData\Loca

Training: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▅▅██
train/ce_loss,█▂▁
train/kd_loss,█▁▁
train/loss,█▁▁
trainer/global_step,▁▁▅▅██
val/ce_loss,█▂▁
val/kd_loss,▁█▆
val/loss,█▃▁
epoch,2
train/ce_loss,1.6018
train/kd_loss,0.21077


In [4]:
from src.train_model_c import train_model_c

model, trainer = train_model_c(
    lr=1e-3,
    batch_size=16,
    epochs=3,
    z_dim=128,
    run_name="modeloC_test_run",
    loss_type="l2"
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type   | Params | Mode 
-----------------------------------------
0 | model | UNetAE | 8.8 M  | train
-----------------------------------------
8.8 M     Trainable params
0         Non-trainable params
8.8 M     Total params
35.148    Total estimated model params size (MB)
21        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▅▅██
train/loss,█▂▁
trainer/global_step,▁▁▅▅██
val/loss,▄█▁
epoch,2
train/loss,0.91334
trainer/global_step,380
val/loss,0.90742


In [5]:
!python src/train_model_a.py model=cnn_scratch logger.wandb.name=run_A_test train.lr=1e-3 trainer.max_epochs=3



Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  3.63it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:13<00:00,  4.86it/s, v_num=cpm1]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 211.75it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 218.25it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 236.62it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 239.10it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00, 12.93it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00, 13.

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 

In [9]:
!python src/train_model_b.py model=distilled logger.wandb.name=run_B_test trainer.max_epochs=3




Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  2.48it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:12<00:00,  4.94it/s, v_num=0rox]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 108.35it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 103.78it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 105.23it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 106.14it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00,  8.28it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00,  9.

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 

In [11]:
!python src/train_model_c.py model=unet_ae logger.wandb.name=run_C_test trainer.max_epochs=3



Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  4.51it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:11<00:00,  5.47it/s, v_num=0jxu]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 338.66it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 307.37it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 307.94it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 312.58it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00, 19.81it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00, 15.

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 

# Entrenamientos

In [12]:
!python src/train_model_a.py model=cnn_scratch logger.wandb.name=A_run1 train.lr=1e-3 trainer.max_epochs=30
!python src/train_model_a.py model=cnn_scratch logger.wandb.name=A_run2 train.lr=5e-4 trainer.max_epochs=50
!python src/train_model_a.py model=cnn_scratch logger.wandb.name=A_run3 train.lr=1e-4 trainer.max_epochs=75

!python src/train_model_b.py model=distilled logger.wandb.name=B_run1 train.lr=1e-4 model.distillation.temperature=4 model.distillation.alpha=0.7 trainer.max_epochs=30
!python src/train_model_b.py model=distilled logger.wandb.name=B_run2 train.lr=5e-5 model.distillation.temperature=3 model.distillation.alpha=0.6 trainer.max_epochs=50
!python src/train_model_b.py model=distilled logger.wandb.name=B_run3 train.lr=1e-5 model.distillation.temperature=5 model.distillation.alpha=0.8 trainer.max_epochs=75

!python src/train_model_c.py model=unet_ae logger.wandb.name=C_run1 train.lr=1e-4 model.latent.dim=128 model.reconstruction.loss=l2 trainer.max_epochs=30
!python src/train_model_c.py model=unet_ae logger.wandb.name=C_run2 train.lr=5e-5 model.latent.dim=256 model.reconstruction.loss=l1 trainer.max_epochs=50
!python src/train_model_c.py model=unet_ae logger.wandb.name=C_run3 train.lr=1e-4 model.latent.dim=64 model.reconstruction.loss=l2 trainer.max_epochs=75



Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  3.01it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:12<00:00,  4.97it/s, v_num=egc3]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 197.19it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 188.89it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 211.76it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 219.45it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00, 11.86it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00, 11.

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 


Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  4.60it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:10<00:00,  5.90it/s, v_num=r40i]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 253.20it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 273.22it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 254.87it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 254.34it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00, 11.84it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00, 13.

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 


Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  2.58it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:12<00:00,  5.02it/s, v_num=3g5t]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 264.57it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 238.12it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 222.84it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 214.64it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00,  9.79it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00, 11.

...

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 


Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  5.03it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:11<00:00,  5.62it/s, v_num=mpzl]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 300.34it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 271.60it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 274.38it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 317.65it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00, 13.99it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00, 16.

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 


Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  4.90it/s]
                                                                           

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 64/64 [00:11<00:00,  5.48it/s, v_num=nnxr]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/7 [00:00<?, ?it/s]

Validation DataLoader 0:  14%|█▍        | 1/7 [00:00<00:00, 325.22it/s]

Validation DataLoader 0:  29%|██▊       | 2/7 [00:00<00:00, 366.14it/s]

Validation DataLoader 0:  43%|████▎     | 3/7 [00:00<00:00, 348.82it/s]

Validation DataLoader 0:  57%|█████▋    | 4/7 [00:00<00:00, 343.82it/s]

Validation DataLoader 0:  71%|███████▏  | 5/7 [00:00<00:00, 16.30it/s] 

Validation DataLoader 0:  86%|████████▌ | 6/7 [00:00<00:00, 19.

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\ianmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly 